# TRANSFORM DATA — Le Panier-Sûr

Nettoyage du CSV brut `champignons_data.csv` vers un CSV structuré `champignons_clean.csv` exploitable par un modèle de classification.

**Étapes :**
1. Chargement + indicateurs de présence (`a_un_pied`, `a_des_pores`, `a_des_lames`)
2. Normalisation (minuscules + suppression des accents)
3. Dictionnaires + extraction binaire (couleurs, textures, odeurs, **saveurs**, **attache des lames**, **morphologie du pied**)
4. Parsing des dimensions (Regex) → `[partie]_taille_min_cm` / `[partie]_taille_max_cm`, NaN → 0
5. Saison → 12 colonnes binaires `saison_mois_01` … `saison_mois_12`
6. Habitat → colonnes binaires (feuillus, conifères, prairies, …)
7. Export : drop des colonnes inutiles (`url`, `image_url`, `categories`, `confusion`, `synonyme`, `nom_scientifique`) et sauvegarde

## 1. Chargement + indicateurs de présence

In [ ]:
import re
import unicodedata
import pandas as pd

RAW_CSV = "../../data/raw/champignons_data.csv"

df = pd.read_csv(RAW_CSV)
print(f"{df.shape[0]} lignes brutes, {df.shape[1]} colonnes")

# --- Indicateurs de présence (Priorité 2) ---
# L'absence d'une partie est une info discriminante :
#   - pas de pied → polypores / lichens
#   - pas de pores → agarics (à lames)
#   - pas de lames → bolets / pézizes
#   - pas de chapeau → champignons résupinés, corail, etc.
#   - pas de chair décrite → structure gélatineuse / absente
# Règle : 1 si la cellule contient du texte utile ; 0 si NaN, vide ou contient "absent".
def has_part(val):
    if not isinstance(val, str):
        return 0
    s = val.strip().lower()
    if not s or "absent" in s:
        return 0
    return 1

PARTS = ["chapeau", "pores", "lames", "pied", "chair"]

df_clean = df.copy()
df_clean["a_un_chapeau"]  = df["chapeau"].apply(has_part)
df_clean["a_des_pores"]   = df["pores"].apply(has_part)
df_clean["a_des_lames"]   = df["lames"].apply(has_part)
df_clean["a_un_pied"]     = df["pied"].apply(has_part)
df_clean["a_de_la_chair"] = df["chair"].apply(has_part)

# --- Filtrage : on supprime les champignons qui n'ont NI chair, NI pied, NI pores, NI lames ---
# Ces lignes n'apportent aucune info exploitable pour la classification.
internal_parts = ["a_de_la_chair", "a_un_pied", "a_des_pores", "a_des_lames"]
mask_keep = df_clean[internal_parts].sum(axis=1) > 0
nb_drop = (~mask_keep).sum()
df       = df.loc[mask_keep].reset_index(drop=True)
df_clean = df_clean.loc[mask_keep].reset_index(drop=True)
print(f"Après filtrage : {len(df_clean)} lignes ({nb_drop} supprimées — aucune partie interne décrite)")

presence_cols = ["a_un_chapeau", "a_des_pores", "a_des_lames", "a_un_pied", "a_de_la_chair"]
print("Présence :", df_clean[presence_cols].sum().to_dict())
df_clean[["nom"] + presence_cols].head(3)

219 lignes brutes, 17 colonnes
Après filtrage : 217 lignes (2 supprimées — aucune partie interne décrite)
Présence : {'a_un_chapeau': 199, 'a_des_pores': 33, 'a_des_lames': 133, 'a_un_pied': 201, 'a_de_la_chair': 215}


,nom,a_un_chapeau,a_des_pores,a_des_lames,a_un_pied,a_de_la_chair
0,AGARIC AUGUSTE,1,0,1,1,1
1,AGARIC DES JACHÈRES,1,0,1,1,1
2,AGARIC JAUNISSANT,1,0,1,1,1


## 2. Normalisation

Passage en minuscules et suppression des accents pour rendre les recherches par mot-clé robustes (évite `forêt` ≠ `foret`, `mèches` ≠ `meches`, etc.).

In [ ]:
def normalize(text):
    """Minuscules + suppression des accents. Renvoie '' si NaN/None."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    # NFD décompose les caractères accentués (é → e + ◌́), on filtre les diacritiques
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text

# Colonnes texte à analyser (confusion est exclue : elle sera droppée à l'export)
TEXT_COLS = ["chapeau", "pores", "lames", "pied", "chair", "odeur", "saveur", "habitat", "saison"]

# On travaille sur une copie normalisée pour l'extraction — le df original reste intact
df_norm = df.copy()
for col in TEXT_COLS:
    df_norm[col] = df_norm[col].apply(normalize)

df_norm[["nom", "chapeau", "saveur", "habitat"]].head(3)

,nom,chapeau,saveur,habitat
0,AGARIC AUGUSTE,"5 a 25 cm, couvert de meches brun-roux sur fon...",douce,"bois clairs de feuillus ou de coniferes, lisie..."
1,AGARIC DES JACHÈRES,"5 a 15 cm, lisse, blanc puis jaunissant en vie...",douce,"surtout sous feuillus, clairieres, lisieres, p..."
2,AGARIC JAUNISSANT,"2 a 15 cm, blanc a grisatre pale, lisse, souve...","desagreable, iodee","prairies, jardins ou bois clairs"


## 3. Dictionnaires de mots-clés

Chaque dictionnaire associe une **étiquette canonique** (nom de colonne) à une liste de **variantes orthographiques** (singulier/pluriel, masculin/féminin, synonymes).

**Pour étendre un dictionnaire :**
- Nouvelle clé = nouvelle colonne binaire → ex. `"turquoise": ["turquoise"]`
- Ajouter une variante à une clé existante → ex. `"brun": [..., "marron", "chatain"]`
- Les variantes doivent être **sans accents** (on cherche dans le texte normalisé)
- Recherche par mots entiers (`\b...\b`) → `rose` ne matche pas dans `rosâtre` (ajouter `"rosatre"` explicitement)

In [ ]:
# --- Couleurs : variantes incluent formes masc/fém, dérivés (-atre) ---
COULEURS = {
    "blanc":  ["blanc", "blanche", "blanchatre", "ivoire", "creme"],
    "brun":   ["brun", "brune", "brunatre", "marron", "chatain", "chocolat", "cannelle"],
    "jaune":  ["jaune", "jaunatre", "jaunissant", "citron", "safran", "miel"],
    "rouge":  ["rouge", "rougeatre", "rougissant", "vineux", "pourpre", "ecarlate"],
    "orange": ["orange", "orangee", "fauve", "cuivre"],
    "rose":   ["rose", "rosatre", "rosee"],
    "gris":   ["gris", "grise", "grisatre", "cendre"],
    "noir":   ["noir", "noire", "noiratre", "noircissant"],
    "vert":   ["vert", "verte", "verdatre", "olive", "olivatre"],
    "bleu":   ["bleu", "bleue", "bleuissant", "bleuatre"],
    "violet": ["violet", "violette", "violace", "lilas"],
    "ocre":   ["ocre", "ocrace", "ocracee", "beige"],
    "roux":   ["roux", "rousse", "roussatre"],
}

# --- Textures / aspect de surface (chapeau, pied) ---
TEXTURES = {
    "lisse":      ["lisse"],
    "meches":     ["meches", "meche"],
    "floconneux": ["floconneux", "floconneuse", "flocons", "pelucheux"],
    "visqueux":   ["visqueux", "visqueuse", "gluant"],
    "velours":    ["velours", "veloute", "velouteuse", "feutre", "feutree"],
    "ecailleux":  ["ecailleux", "ecailleuse", "ecailles"],
    "craquele":   ["craquele", "craquelee", "craquelure"],
    "strie":      ["strie", "striee", "stries", "raye", "rayures"],
}

# --- Odeurs ---
ODEURS = {
    "anise":       ["anis", "anisee", "anise"],
    "phenol":      ["phenol", "encre"],
    "iode":        ["iode", "iodee", "marine", "maree"],
    "radis":       ["radis"],
    "farine":      ["farine", "farineuse"],
    "amande":      ["amande", "amandes"],
    "terre":       ["terre", "terreuse", "terreux"],
    "fruitee":     ["fruit", "fruitee", "pomme"],
    "desagreable": ["desagreable"],
}

# --- Saveurs (Priorité 3) ---
SAVEURS = {
    "douce":       ["douce", "doux", "agreable"],
    "amere":       ["amere", "amer", "amertume"],
    "acide":       ["acide", "acidulee", "aigre"],
    "piquante":    ["piquante", "piquant", "brulante"],
    "poivree":     ["poivree", "poivre"],
    "iodee":       ["iodee", "iode"],
    "desagreable": ["desagreable"],
    "sans_saveur": ["sans"],
}

# --- Attache (lames ET pores : adnées, décurrentes, libres…) ---
ATTACHE = {
    "libres":      ["libres", "libre"],
    "adnees":      ["adnees", "adnee", "adne", "adnes"],
    "decurrentes": ["decurrentes", "decurrente", "decurrent", "decurrents"],
    "echancrees":  ["echancrees", "echancree", "echancre", "echancres"],
    "serrees":     ["serrees", "serree"],
    "espacees":    ["espacees", "espacee", "ecartees"],
}

# --- Morphologie du pied (anneau/volve/bulbe → déterminant pour amanites) ---
PIED_MORPHO = {
    "anneau":      ["anneau", "anneaux"],
    "volve":       ["volve", "volves"],
    "bulbe":       ["bulbe", "bulbeux", "bulbeuse"],
    "massue":      ["massue"],
    "creux":       ["creux", "creuse"],
    "elance":      ["elance", "elancee"],
    "cylindrique": ["cylindrique"],
    "reseau":      ["reseau", "reticule"],
}

# --- Consistance de la chair (texture interne, pas de surface) ---
CHAIR_CONSISTANCE = {
    "ferme":     ["ferme"],
    "tendre":    ["tendre"],
    "molle":     ["molle", "mou"],
    "cassante":  ["cassante", "cassant"],
    "elastique": ["elastique"],
    "epaisse":   ["epaisse", "epais", "charnue", "charnu"],
    "fibreuse":  ["fibreuse", "fibreux"],
    "spongieuse": ["spongieuse", "spongieux"],
}

print(f"{len(COULEURS)} couleurs, {len(TEXTURES)} textures, {len(ODEURS)} odeurs, "
      f"{len(SAVEURS)} saveurs, {len(ATTACHE)} attaches, {len(PIED_MORPHO)} pied_morpho, "
      f"{len(CHAIR_CONSISTANCE)} chair_consistance")

13 couleurs, 8 textures, 9 odeurs, 8 saveurs, 6 attaches, 8 pied_morpho, 8 chair_consistance


### 3.b. Application des dictionnaires aux colonnes texte

Pour chaque colonne et chaque dictionnaire du plan, on crée des colonnes `{partie}_{dico}_{etiquette}` valant `1` si au moins une variante est trouvée, `0` sinon.

In [ ]:
def contains_any(text, variants):
    """1 si l'un des mots entiers de `variants` est présent dans `text`, 0 sinon."""
    if not text:
        return 0
    # \b = frontière de mot → évite que "rose" matche dans "rosâtre"
    pattern = r"\b(?:" + "|".join(re.escape(v) for v in variants) + r")\b"
    return int(bool(re.search(pattern, text)))

def extract_dict_features(df_src, text_col, dictionary, dict_name):
    """Ajoute une colonne binaire par étiquette, préfixée par `{text_col}_{dict_name}_`."""
    features = {}
    for label, variants in dictionary.items():
        col_name = f"{text_col}_{dict_name}_{label}"
        features[col_name] = df_src[text_col].apply(lambda t, v=variants: contains_any(t, v))
    return pd.DataFrame(features)

# Plan d'extraction : chaque colonne (chapeau/pores/lames/pied/chair) reçoit
# la couleur + un ou plusieurs dictionnaires spécifiques à sa sémantique.
# Pour ajouter un dictionnaire à une colonne : y ajouter un tuple (prefix, DICT).
EXTRACTION_PLAN = {
    "chapeau": [("couleur", COULEURS), ("texture", TEXTURES)],
    "pores":   [("couleur", COULEURS), ("attache", ATTACHE)],
    "lames":   [("couleur", COULEURS), ("attache", ATTACHE)],
    "pied":    [("couleur", COULEURS), ("texture", TEXTURES), ("morpho", PIED_MORPHO)],
    "chair":   [("couleur", COULEURS), ("consistance", CHAIR_CONSISTANCE)],
    "odeur":   [("type", ODEURS)],
    "saveur":  [("type", SAVEURS)],
}

for col, plan in EXTRACTION_PLAN.items():
    for dict_name, dictionary in plan:
        feats = extract_dict_features(df_norm, col, dictionary, dict_name)
        df_clean = pd.concat([df_clean, feats], axis=1)

print(f"Après extraction dictionnaires : {df_clean.shape[1]} colonnes")
df_clean.filter(regex="^(chair_consistance_|pores_attache_)").head(3)

Après extraction dictionnaires : 148 colonnes


,pores_attache_libres,pores_attache_adnees,pores_attache_decurrentes,pores_attache_echancrees,pores_attache_serrees,pores_attache_espacees,chair_consistance_ferme,chair_consistance_tendre,chair_consistance_molle,chair_consistance_cassante,chair_consistance_elastique,chair_consistance_epaisse,chair_consistance_fibreuse,chair_consistance_spongieuse
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 4. Parsing des dimensions (Regex)

Les descriptions contiennent des tailles exprimées en centimètres ou millimètres, typiquement sous la forme :
- `"5 à 25 cm"` → min=5, max=25
- `"3 cm"` → min=max=3
- `"7 à 15 cm"` / `"Élancé (5 à 12 cm)"` → min=5, max=12

**Règle :** on ne capture un nombre que s'il est **suivi** (directement ou après une plage) d'une unité de longueur (`cm`, `mm`, `centimetres`, `millimetres`). Cela évite de capturer des nombres sans contexte (ex. nombre de lamelles).

Les millimètres sont convertis en cm (÷10) pour uniformiser l'unité.

In [ ]:
# Regex :
#   - (\d+(?:[.,]\d+)?)             → premier nombre (entier ou décimal avec . ou ,)
#   - (?:\s*(?:a|-)\s*(\d+(?:[.,]\d+)?))?  → optionnel : " à X " ou " - X " pour une plage
#   - \s*(cm|mm|centimetres?|millimetres?) → unité obligatoire juste après
# Après normalisation, "à" devient "a" — le pattern reflète le texte normalisé.
SIZE_REGEX = re.compile(
    r"(\d+(?:[.,]\d+)?)\s*(?:a|-)?\s*(\d+(?:[.,]\d+)?)?\s*(cm|mm|centimetres?|millimetres?)"
)

def parse_sizes(text):
    """Extrait (min_cm, max_cm) depuis un texte normalisé. Renvoie (None, None) si rien trouvé."""
    if not text:
        return (None, None)
    match = SIZE_REGEX.search(text)
    if not match:
        return (None, None)
    n1, n2, unit = match.groups()
    to_float = lambda s: float(s.replace(",", "."))
    vmin = to_float(n1)
    vmax = to_float(n2) if n2 else vmin
    if unit.startswith("mm") or unit.startswith("millimetre"):
        vmin, vmax = vmin / 10, vmax / 10
    return (vmin, vmax)

SIZE_COLS = ["chapeau", "pied"]  # seules ces parties portent des tailles dans le CSV brut
for col in SIZE_COLS:
    sizes = df_norm[col].apply(parse_sizes)
    # NaN → 0 : si la partie est absente ou sans taille, on considère la taille nulle (Priorité 4)
    df_clean[f"{col}_taille_min_cm"] = sizes.apply(lambda t: t[0]).fillna(0)
    df_clean[f"{col}_taille_max_cm"] = sizes.apply(lambda t: t[1]).fillna(0)

df_clean[["nom", "chapeau_taille_min_cm", "chapeau_taille_max_cm",
          "pied_taille_min_cm", "pied_taille_max_cm"]].head(5)

,nom,chapeau_taille_min_cm,chapeau_taille_max_cm,pied_taille_min_cm,pied_taille_max_cm
0,AGARIC AUGUSTE,5.0,25.0,6.0,20.0
1,AGARIC DES JACHÈRES,5.0,15.0,5.0,15.0
2,AGARIC JAUNISSANT,2.0,15.0,3.0,15.0
3,AGARIC SYLVICOLE,3.0,12.0,2.0,15.0
4,AMANITE CITRINE,4.0,10.0,3.0,15.0


## 5. Saison → 12 colonnes binaires (une par mois)

La colonne `saison` suit le format `"Juillet > Octobre"`. Pour un modèle de classification, on préfère un encodage binaire par mois (12 colonnes `saison_mois_01` … `saison_mois_12`) plutôt que deux entiers début/fin : cela évite de donner une structure ordinale artificielle aux mois et gère correctement les saisons qui chevauchent deux années (ex. `novembre > fevrier` → mois 11, 12, 1, 2).

In [ ]:
MOIS = {
    "janvier": 1, "fevrier": 2, "mars": 3, "avril": 4, "mai": 5, "juin": 6,
    "juillet": 7, "aout": 8, "septembre": 9, "octobre": 10, "novembre": 11, "decembre": 12,
}

def mois_actifs(text):
    """Renvoie l'ensemble des mois (1-12) couverts par 'mois1 > mois2'.
    Gère le wrap (novembre > fevrier → {11, 12, 1, 2})."""
    if not text:
        return set()
    parts = re.split(r"\s*>\s*", text)
    if len(parts) != 2:
        return set()
    debut = MOIS.get(parts[0].strip())
    fin = MOIS.get(parts[1].strip())
    if debut is None or fin is None:
        return set()
    if debut <= fin:
        return set(range(debut, fin + 1))
    # wrap-around : debut=11, fin=2 → 11, 12, 1, 2
    return set(range(debut, 13)) | set(range(1, fin + 1))

actifs = df_norm["saison"].apply(mois_actifs)
for m in range(1, 13):
    df_clean[f"saison_mois_{m:02d}"] = actifs.apply(lambda s, m=m: int(m in s))

df_clean[["nom", "saison"] + [f"saison_mois_{m:02d}" for m in range(1, 13)]].head(5)

,nom,saison,saison_mois_01,saison_mois_02,saison_mois_03,saison_mois_04,saison_mois_05,saison_mois_06,saison_mois_07,saison_mois_08,saison_mois_09,saison_mois_10,saison_mois_11,saison_mois_12
0,AGARIC AUGUSTE,Juillet > Octobre,0,0,0,0,0,0,1,1,1,1,0,0
1,AGARIC DES JACHÈRES,Juin > Novembre,0,0,0,0,0,1,1,1,1,1,1,0
2,AGARIC JAUNISSANT,Mai > Novembre,0,0,0,0,1,1,1,1,1,1,1,0
3,AGARIC SYLVICOLE,Août > Octobre,0,0,0,0,0,0,0,1,1,1,0,0
4,AMANITE CITRINE,Juillet > Octobre,0,0,0,0,0,0,1,1,1,1,0,0


## 6. Habitat → colonnes binaires

Le champ `habitat` mélange types de milieu (prairies, forêt, clairière) et essences d'arbres (chênes, pins, hêtres). On produit une colonne binaire par étiquette — même mécanisme que pour les couleurs/textures.

In [ ]:
# Pour ajouter un nouvel habitat : nouvelle entrée dans le dict (variantes sans accents).
HABITATS = {
    "feuillus":     ["feuillus", "feuillu"],
    "coniferes":    ["coniferes", "conifere"],
    "foret":        ["foret", "forets"],
    "prairies":     ["prairies", "prairie", "pelouse", "pelouses", "paturages"],
    "clairieres":   ["clairiere", "clairieres"],
    "lisieres":     ["lisiere", "lisieres"],
    "jardins":      ["jardin", "jardins", "parc", "parcs"],
    "bois":         ["bois"],
    "chenes":       ["chenes", "chene"],
    "hetres":       ["hetres", "hetre"],
    "pins":         ["pins", "pin"],
    "bouleaux":     ["bouleaux", "bouleau"],
    "chataigniers": ["chataigniers", "chataignier"],
    "charmes":      ["charmes", "charme"],
    "melezes":      ["melezes", "meleze"],
    "bois_morts":   ["morts", "mort"],
}

habitat_feats = extract_dict_features(df_norm, "habitat", HABITATS, "type")
df_clean = pd.concat([df_clean, habitat_feats], axis=1)

print(f"Après habitat : {df_clean.shape[1]} colonnes")
df_clean.filter(like="habitat_type_").head(3)

Après habitat : 180 colonnes


,habitat_type_feuillus,habitat_type_coniferes,habitat_type_foret,habitat_type_prairies,habitat_type_clairieres,habitat_type_lisieres,habitat_type_jardins,habitat_type_bois,habitat_type_chenes,habitat_type_hetres,habitat_type_pins,habitat_type_bouleaux,habitat_type_chataigniers,habitat_type_charmes,habitat_type_melezes,habitat_type_bois_morts
0,1,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0
1,1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0


## 7. Export

Deux versions sont sauvegardées :
- `data/clean_train_data/champignons_clean.csv` → **données pour l'entraînement** : `nom`, `statut` + toutes les features binaires/numériques (sans les colonnes texte brut).
- `data/transform/champignons_raw_text.csv` → **données texte d'origine** : `nom`, `statut` + les colonnes descriptives (`chapeau`, `pores`, `lames`, `pied`, `chair`, `odeur`, `saveur`, `habitat`, `saison`).

In [ ]:
import os

TRAIN_DIR = "../../data/clean_train_data"
RAW_DIR   = "../../data/transform"
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

# Priorité 1 : drop des colonnes non exploitables pour l'apprentissage.
#   - url, image_url       : identifiants / liens
#   - nom_scientifique     : fuite de label (1:1 avec nom)
#   - synonyme             : fuite de label + faible couverture
#   - confusion            : donne la réponse (liste des espèces proches)
#   - categories           : non utilisée
DROP_COLS = ["url", "image_url", "categories", "confusion", "synonyme", "nom_scientifique"]
df_clean = df_clean.drop(columns=[c for c in DROP_COLS if c in df_clean.columns])

# Colonnes texte brut (descriptions d'origine) à séparer des features encodées.
RAW_TEXT_COLS = ["chapeau", "pores", "lames", "pied", "chair",
                 "odeur", "saveur", "habitat", "saison"]

# --- Version 1 : entraînement (nom + statut + features binaires/numériques) ---
train_cols = [c for c in df_clean.columns if c not in RAW_TEXT_COLS]
df_train = df_clean[train_cols]
train_csv = f"{TRAIN_DIR}/champignons_clean.csv"
df_train.to_csv(train_csv, index=False)
print(f"[train] {len(df_train)} lignes, {len(df_train.columns)} colonnes → {train_csv}")

# --- Version 2 : texte d'origine (nom + statut + colonnes descriptives) ---
raw_cols = ["nom", "statut"] + [c for c in RAW_TEXT_COLS if c in df_clean.columns]
df_raw = df_clean[raw_cols]
raw_csv = f"{RAW_DIR}/champignons_raw_text.csv"
df_raw.to_csv(raw_csv, index=False)
print(f"[raw]   {len(df_raw)} lignes, {len(df_raw.columns)} colonnes → {raw_csv}")

[train] 217 lignes, 165 colonnes → ../../data/clean_train_data/champignons_clean.csv
[raw]   217 lignes, 11 colonnes → ../../data/transform/champignons_raw_text.csv
